In [1]:
"""
NOTEBOOK D -- ANALYSIS (Cells 9-11 combined)
============================================================================
Self-contained. CPU only -- no GPU needed, no model loading. ~10 minutes.

Replaces the three-cell sequence with one script. Nothing from
stage9_final.py or step1_generation_adapted.py needs to be in scope: every
loader and metric this needs is defined here.

DESIGN DECISION THAT MATTERS: PAIRING BY BASELINE CLASS ASSIGNMENT
The fine-tuned chunks hold both classes mixed together, and first_match()
can file a multi-label image under a different class than the baseline run
did. If both arms were classified independently, those images would pair
against the wrong boxes or silently drop.

So this script does NOT re-derive class membership for the fine-tuned arm.
It iterates over the BASELINE per-image rows and, for each
(class_name, image_id) already scored in the baseline, computes the
fine-tuned metrics for that SAME class. Pairing is exact by construction
and the box masks are identical across arms.

That is also why you may see fewer than 150 pairs: an image generated in
the fine-tuned arm but filed under a different class in the baseline has no
counterpart, and is reported as unpaired rather than force-matched.

INPUT DATASETS TO ATTACH
  1. Cell 7 output          -- the fine-tuned maps (uncertainty_chunk_*.npz)
  2. vindr-cxr-512          -- H5 archive + vindr_cxr_test.csv
  3. annotation CSV         -- the file with the `boxes` column
  4. baseline results       -- edge_correlation_baseline.csv
                               stage9_per_image_metrics.csv
"""

import ast
import json
from ast import literal_eval
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
from scipy.ndimage import sobel, zoom
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score, average_precision_score
from statsmodels.stats.multitest import multipletests
from tqdm.auto import tqdm

# =============================================================================
# CONFIG -- set every CHANGE-ME
# =============================================================================
FINETUNED_MAP_DIR = "/kaggle/input/datasets/kartikichandratre/ext-step7-lora-rank8/finetuned_maps"

H5_PATH        = "/kaggle/input/datasets/pgc17ms072/vindr-cxr-512-h5/vindr_cxr_512.h5"
ANNOTATION_CSV = "/kaggle/input/datasets/pgc17ms072/vindr-cxr-512-h5/vindr_cxr_metadata.csv"
TEST_META_CSV  = "/kaggle/input/datasets/pgc17ms072/vindr-cxr-512-h5/vindr_cxr_test.csv"

BASELINE_EDGE_CSV      = "/kaggle/input/datasets/kartikichandratre/ext-step0-edge-correlation/edge_correlation_baseline.csv"
BASELINE_PER_IMAGE_CSV = "/kaggle/input/datasets/kartikichandratre/stage9-results/stage9_per_image_metrics.csv"

OUT_DIR = Path("/kaggle/working/analysis")

MAP_SIZE = 512
N_BOOTSTRAP = 2000
RANDOM_SEED = 42
ALPHA = 0.05
DICE_PERCENTILES = [90.0, 95.0, 99.0]

# Box coords are in original DICOM pixels; width/height columns are named
# 'columns'/'rows' in this dataset (not 'width'/'height').
ORIGINAL_SIZE_COLS = {"width": "columns", "height": "rows"}

# PRE-REGISTERED. Fitted on baseline data only, before fine-tuning existed.
# Refitting these on combined data would be circular -- do not.
BASELINE_SLOPES = {
    "Atelectasis":   0.2055,
    "Cardiomegaly":  0.1914,
    "Consolidation": 0.2867,
    "Nodule/Mass":   0.2568,
    "Pneumothorax":  0.5058,
}

CHUNK_SEPARATOR = "__"
CHUNK_VALUE_SUFFIXES = ["variance", "var", "uncertainty", "sigma2"]
CHUNK_IGNORE_SUFFIXES = ["mean", "std_err", "count", "n", "z_t"]


# =============================================================================
# Loaders
# =============================================================================
def load_map_dir(d):
    """Chunked .npz -> {image_id: variance map at MAP_SIZE}."""
    out = {}
    for f in sorted(Path(d).glob("*.npz")):
        with np.load(f, allow_pickle=True) as z:
            for key in z.files:
                if CHUNK_SEPARATOR not in key:
                    continue
                image_id, _, suffix = key.rpartition(CHUNK_SEPARATOR)
                s = suffix.lower()
                if s in CHUNK_IGNORE_SUFFIXES or s not in CHUNK_VALUE_SUFFIXES:
                    continue
                m = np.squeeze(np.array(z[key])).astype(np.float64)
                if m.ndim == 3:
                    m = m.var(axis=0)
                if m.shape != (MAP_SIZE, MAP_SIZE):
                    m = zoom(m, (MAP_SIZE / m.shape[0], MAP_SIZE / m.shape[1]),
                             order=1)
                out[image_id] = m
    return out


def explode_boxes(df):
    """One row per image with a packed `boxes` column -> one row per box."""
    records = []
    for _, row in df.iterrows():
        raw = row.get("boxes")
        if raw is None or (isinstance(raw, float) and pd.isna(raw)):
            continue
        try:
            boxes = literal_eval(raw) if isinstance(raw, str) else raw
        except (ValueError, SyntaxError):
            continue
        for b in boxes:
            records.append({"image_id": row["image_id"],
                            "class_name": b.get("class_name"),
                            "x_min": b.get("x_min"), "y_min": b.get("y_min"),
                            "x_max": b.get("x_max"), "y_max": b.get("y_max")})
    return pd.DataFrame(records)


def build_box_mask(annot_long, sizes, class_name, image_id):
    """Union of all readers' boxes for this class, rasterised to MAP_SIZE."""
    rows = annot_long[(annot_long.class_name == class_name)
                      & (annot_long.image_id.astype(str) == str(image_id))]
    if rows.empty or str(image_id) not in sizes:
        return None
    ow, oh = sizes[str(image_id)]
    sx, sy = MAP_SIZE / ow, MAP_SIZE / oh
    mask = np.zeros((MAP_SIZE, MAP_SIZE), dtype=bool)
    for _, r in rows.iterrows():
        if any(pd.isna(r[c]) for c in ["x_min", "y_min", "x_max", "y_max"]):
            continue
        x0 = int(np.clip(round(r.x_min * sx), 0, MAP_SIZE - 1))
        x1 = int(np.clip(round(r.x_max * sx), 0, MAP_SIZE))
        y0 = int(np.clip(round(r.y_min * sy), 0, MAP_SIZE - 1))
        y1 = int(np.clip(round(r.y_max * sy), 0, MAP_SIZE))
        if x1 > x0 and y1 > y0:
            mask[y0:y1, x0:x1] = True
    return mask if (mask.any() and not mask.all()) else None


# =============================================================================
# Metrics
# =============================================================================
def dice(binary_map, mask):
    inter = np.logical_and(binary_map, mask).sum()
    denom = binary_map.sum() + mask.sum()
    return float(2 * inter / denom) if denom else np.nan


def metrics_for_image(var_map, mask):
    y = mask.ravel().astype(np.uint8)
    s = np.nan_to_num(var_map.ravel(), nan=0.0, posinf=0.0, neginf=0.0)
    if y.sum() == 0 or y.sum() == len(y):
        return None
    row = {"pixel_auroc": float(roc_auc_score(y, s)),
           "pixel_auprc": float(average_precision_score(y, s)),
           "box_fraction": float(y.mean())}
    row["auprc_lift"] = row["pixel_auprc"] / row["box_fraction"]
    for p in DICE_PERCENTILES:
        row[f"dice_p{p:g}"] = dice(var_map >= np.percentile(var_map, p), mask)
    row["pointing_hit"] = int(mask.ravel()[int(np.argmax(s))])
    return row


def edge_variance_correlation(img, var_map):
    """Spearman rho between Sobel edge magnitude and the variance map."""
    img = img.astype(np.float64)
    edges = np.hypot(sobel(img, axis=0), sobel(img, axis=1))
    return float(spearmanr(edges.ravel(), var_map.ravel())[0])


def bootstrap_mean_ci(v, n_boot=N_BOOTSTRAP, seed=RANDOM_SEED):
    v = np.asarray([x for x in v if np.isfinite(x)], dtype=float)
    if len(v) < 3:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    means = np.array([rng.choice(v, len(v), replace=True).mean()
                      for _ in range(n_boot)])
    return float(v.mean()), float(np.percentile(means, 2.5)), \
           float(np.percentile(means, 97.5))


# =============================================================================
# PART 1 -- score the fine-tuned arm, paired to baseline class assignment
# =============================================================================
def score_finetuned():
    print("[1/4] Loading inputs...")
    ft_maps = load_map_dir(FINETUNED_MAP_DIR)
    print(f"  fine-tuned maps: {len(ft_maps)}")

    base_per = pd.read_csv(BASELINE_PER_IMAGE_CSV)
    base_per = base_per[base_per.method == "diffusion"]
    print(f"  baseline per-image rows: {len(base_per)}")

    annot_long = explode_boxes(pd.read_csv(ANNOTATION_CSV))
    print(f"  annotation boxes: {len(annot_long)}")

    meta = pd.read_csv(TEST_META_CSV)
    w, h = ORIGINAL_SIZE_COLS["width"], ORIGINAL_SIZE_COLS["height"]
    assert w in meta.columns and h in meta.columns, (
        f"TEST_META_CSV lacks '{w}'/'{h}'. Columns: {meta.columns.tolist()}")
    sizes = {str(r.image_id): (float(r[w]), float(r[h]))
             for _, r in meta.iterrows()}

    # Pair on baseline's (class_name, image_id). See module docstring.
    pairs = base_per[base_per.image_id.astype(str).isin(ft_maps.keys())]
    print(f"\n  pairable (in both arms): {len(pairs)}")
    print(pairs.class_name.value_counts().to_string())

    unpaired = set(ft_maps.keys()) - set(base_per.image_id.astype(str))
    if unpaired:
        print(f"\n  {len(unpaired)} fine-tuned maps have NO baseline row "
              f"(dropped): {sorted(unpaired)[:3]}")

    print("\n[2/4] Scoring fine-tuned maps...")
    rows = []
    with h5py.File(H5_PATH, "r") as h5f:
        for _, br in tqdm(pairs.iterrows(), total=len(pairs)):
            image_id, class_name = str(br.image_id), br.class_name
            mask = build_box_mask(annot_long, sizes, class_name, image_id)
            if mask is None:
                continue
            var = ft_maps[image_id]
            m = metrics_for_image(var, mask)
            if m is None:
                continue
            m["edge_rho"] = edge_variance_correlation(h5f[image_id][:], var)
            m.update({"class_name": class_name, "image_id": image_id})
            rows.append(m)

    ft = pd.DataFrame(rows)
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    ft.to_csv(OUT_DIR / "finetuned_per_image_metrics.csv", index=False)
    print(f"  scored {len(ft)} images")
    return ft


# =============================================================================
# PART 2 -- the calibration test
# =============================================================================
def calibration(ft):
    print("\n[3/4] Building paired frame...")
    be = pd.read_csv(BASELINE_EDGE_CSV)[["class_name", "image_id", "edge_rho"]]
    bp = pd.read_csv(BASELINE_PER_IMAGE_CSV)
    bp = bp[bp.method == "diffusion"][
        ["class_name", "image_id", "pixel_auroc", "pointing_hit", "box_fraction"]]
    base = be.merge(bp, on=["class_name", "image_id"])

    base["image_id"] = base.image_id.astype(str)
    ft = ft.copy()
    ft["image_id"] = ft.image_id.astype(str)

    m = base.merge(ft[["class_name", "image_id", "edge_rho", "pixel_auroc",
                       "pointing_hit"]],
                   on=["class_name", "image_id"], suffixes=("_base", "_ft"))
    print(f"  paired on {len(m)} images")
    print(m.class_name.value_counts().to_string())
    if len(m) == 0:
        raise RuntimeError("Zero pairs. Check that class_name matches across "
                           "arms and that image_ids are the same dtype.")
    # ---- FILTER TO PROMPT-MATCHED PAIRS ----------------------------------
    # Baseline maps live in separate per-class datasets, each generated with
    # its own prompt ("a chest x-ray showing consolidation" etc). The
    # fine-tuned run produced ONE map per image, with first_match picking a
    # single prompt. For a multi-label image the two arms can therefore
    # differ in prompt, and that difference would show up as a LoRA effect.
    # Keep only pairs where the fine-tuned prompt matches the baseline class.
    meta_files = sorted(Path(FINETUNED_MAP_DIR).glob("*_meta.csv"))
    assert meta_files, f"no *_meta.csv in {FINETUNED_MAP_DIR}"
    gen_meta = pd.concat([pd.read_csv(p) for p in meta_files])
    gen_meta["image_id"] = gen_meta.image_id.astype(str)

    before = len(m)
    m = m.merge(gen_meta[["image_id", "finding"]], on="image_id", how="left")
    m = m[m.finding == m.class_name].drop(columns=["finding"]).reset_index(drop=True)

    print(f"\n  prompt-matched filter: {before} -> {len(m)} pairs")
    print(m.class_name.value_counts().to_string())
    assert len(m) > 0, "no prompt-matched pairs -- check the meta CSV finding column"
    # ----------------------------------------------------------------------
    

    m["delta_rho"] = m.edge_rho_ft - m.edge_rho_base
    m["delta_auroc_obs"] = m.pixel_auroc_ft - m.pixel_auroc_base
    m["slope"] = m.class_name.map(BASELINE_SLOPES)
    m["delta_auroc_pred"] = m.slope * m.delta_rho
    m["residual"] = m.delta_auroc_obs - m.delta_auroc_pred

    print("\n[4/4] Analyses\n")
    print("=" * 74)
    print("1. DID FINE-TUNING MOVE EDGE-DOMINANCE?")
    print("=" * 74)
    r = []
    for c, g in m.groupby("class_name"):
        _, p = stats.wilcoxon(g.edge_rho_ft, g.edge_rho_base)
        mu, lo, hi = bootstrap_mean_ci(g.delta_rho)
        r.append({"class_name": c, "n": len(g),
                  "rho_base": g.edge_rho_base.mean(),
                  "rho_ft": g.edge_rho_ft.mean(),
                  "delta_rho": mu, "ci_low": lo, "ci_high": hi, "p": p})
    t1 = pd.DataFrame(r)
    t1["p_bh"] = multipletests(t1.p, alpha=ALPHA, method="fdr_bh")[1]
    print(t1.round(4).to_string(index=False))

    print("\n" + "=" * 74)
    print("2. DID LOCALISATION CHANGE?")
    print("=" * 74)
    r = []
    for c, g in m.groupby("class_name"):
        _, p = stats.wilcoxon(g.pixel_auroc_ft, g.pixel_auroc_base)
        mu, lo, hi = bootstrap_mean_ci(g.delta_auroc_obs)
        r.append({"class_name": c, "n": len(g),
                  "auroc_base": g.pixel_auroc_base.mean(),
                  "auroc_ft": g.pixel_auroc_ft.mean(),
                  "delta_obs": mu, "ci_low": lo, "ci_high": hi, "p": p})
    t2 = pd.DataFrame(r)
    t2["p_bh"] = multipletests(t2.p, alpha=ALPHA, method="fdr_bh")[1]
    print(t2.round(4).to_string(index=False))

    print("\n" + "=" * 74)
    print("3. CALIBRATION TEST -- observed vs pre-registered prediction")
    print("=" * 74)
    r = []
    for c, g in m.groupby("class_name"):
        _, p = stats.wilcoxon(g.residual)
        rc, _ = spearmanr(g.delta_auroc_pred, g.delta_auroc_obs)
        mu, lo, hi = bootstrap_mean_ci(g.residual)
        r.append({"class_name": c, "n": len(g), "slope": g.slope.iloc[0],
                  "pred": g.delta_auroc_pred.mean(),
                  "obs": g.delta_auroc_obs.mean(),
                  "residual": mu, "res_ci_low": lo, "res_ci_high": hi,
                  "corr_pred_obs": rc, "p_residual": p})
    t3 = pd.DataFrame(r)
    t3["p_bh"] = multipletests(t3.p_residual, alpha=ALPHA, method="fdr_bh")[1]
    print(t3.round(4).to_string(index=False))

    print("\nREADING ROW BY ROW:")
    print("  residual ~ 0, p_bh > 0.05  -> the edge/anatomy relationship")
    print("      explains the whole effect. Bulk-prior mechanism CONFIRMED.")
    print("  residual > 0, p_bh < 0.05  -> gain BEYOND edge structure:")
    print("      genuine pathology sensitivity from fine-tuning.")
    print("  residual < 0, p_bh < 0.05  -> fine-tuning cost something the")
    print("      edge relationship does not capture.")

    print("\n" + "=" * 74)
    print("4. POINTING-HIT -- does peak variance ever land in a box?")
    print("=" * 74)
    hb, hf = int(m.pointing_hit_base.sum()), int(m.pointing_hit_ft.sum())
    print(f"  baseline:   {hb} / {len(m)}")
    print(f"  fine-tuned: {hf} / {len(m)}")
    if hb == 0 and hf == 0:
        print("  Still exactly zero. The strongest single statement in the")
        print("  thesis survives domain adaptation.")
    elif hf != hb:
        print(f"  Moved by {hf - hb:+d}. Report the exact count, not a rate.")

    m.to_csv(OUT_DIR / "calibration_paired.csv", index=False)
    t1.to_csv(OUT_DIR / "t1_edge_shift.csv", index=False)
    t2.to_csv(OUT_DIR / "t2_auroc_shift.csv", index=False)
    t3.to_csv(OUT_DIR / "t3_calibration.csv", index=False)

    summary = {
        "n_paired": int(len(m)),
        "per_class_n": m.class_name.value_counts().to_dict(),
        "pointing_hit_baseline": hb, "pointing_hit_finetuned": hf,
        "mean_delta_rho": float(m.delta_rho.mean()),
        "mean_delta_auroc_obs": float(m.delta_auroc_obs.mean()),
        "mean_delta_auroc_pred": float(m.delta_auroc_pred.mean()),
        "mean_residual": float(m.residual.mean()),
        "pre_registered_slopes": BASELINE_SLOPES,
    }
    (OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=2))
    return m, t1, t2, t3


# =============================================================================
# Figure
# =============================================================================
def plot(m, t3):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))

    ax = axes[0]
    for c, g in m.groupby("class_name"):
        ax.scatter(g.edge_rho_base, g.edge_rho_ft, s=14, alpha=0.6, label=c)
    lim = [min(m.edge_rho_base.min(), m.edge_rho_ft.min()) - 0.05,
           max(m.edge_rho_base.max(), m.edge_rho_ft.max()) + 0.05]
    ax.plot(lim, lim, "k--", lw=1)
    ax.set_xlabel(r"edge $\rho$, baseline")
    ax.set_ylabel(r"edge $\rho$, fine-tuned")
    ax.set_title("Did fine-tuning move edge-dominance?")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    ax = axes[1]
    for c, g in m.groupby("class_name"):
        ax.scatter(g.delta_auroc_pred, g.delta_auroc_obs, s=14, alpha=0.6,
                   label=c)
    lo = min(m.delta_auroc_pred.min(), m.delta_auroc_obs.min())
    hi = max(m.delta_auroc_pred.max(), m.delta_auroc_obs.max())
    ax.plot([lo, hi], [lo, hi], "k--", lw=1, label="perfect calibration")
    ax.axhline(0, color="grey", lw=0.6); ax.axvline(0, color="grey", lw=0.6)
    ax.set_xlabel(r"predicted $\Delta$AUROC (slope $\times\ \Delta\rho$)")
    ax.set_ylabel(r"observed $\Delta$AUROC")
    ax.set_title("Calibration test")
    ax.legend(fontsize=7); ax.grid(alpha=0.3)

    ax = axes[2]
    x = np.arange(len(t3))
    t3m = t3.merge(t2[["class_name", "ci_low", "ci_high"]], on="class_name")
    ax.bar(x - 0.2, t3m.pred, 0.4, label="predicted", color="#888888")
    ax.bar(x + 0.2, t3m.obs, 0.4, label="observed", color="#a5453b")
    ax.errorbar(x + 0.2, t3m.obs,
                yerr=[np.abs(t3m.obs - t3m.ci_low),
                      np.abs(t3m.ci_high - t3m.obs)],
                fmt="none", ecolor="k", capsize=3, lw=0.8)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(t3.class_name, rotation=15, ha="right", fontsize=8)
    ax.set_ylabel(r"mean $\Delta$AUROC")
    ax.set_title("Predicted vs observed by class")
    ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

    fig.tight_layout()
    fig.savefig(OUT_DIR / "calibration.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"\nsaved {OUT_DIR / 'calibration.png'}")


if __name__ == "__main__":
    ft = score_finetuned()
    m, t1, t2, t3 = calibration(ft)
    plot(m, t3)
    print(f"\nall outputs -> {OUT_DIR}")

[1/4] Loading inputs...
  fine-tuned maps: 150
  baseline per-image rows: 547
  annotation boxes: 36096

  pairable (in both arms): 207
class_name
Cardiomegaly     107
Consolidation     54
Nodule/Mass       34
Atelectasis        8
Pneumothorax       4

[2/4] Scoring fine-tuned maps...


  0%|          | 0/207 [00:00<?, ?it/s]

  scored 207 images

[3/4] Building paired frame...
  paired on 207 images
class_name
Cardiomegaly     107
Consolidation     54
Nodule/Mass       34
Atelectasis        8
Pneumothorax       4

  prompt-matched filter: 207 -> 150 pairs
class_name
Cardiomegaly     107
Consolidation     43

[4/4] Analyses

1. DID FINE-TUNING MOVE EDGE-DOMINANCE?
   class_name   n  rho_base  rho_ft  delta_rho  ci_low  ci_high   p  p_bh
 Cardiomegaly 107    0.4844  0.5299     0.0455  0.0396   0.0515 0.0   0.0
Consolidation  43    0.4504  0.5117     0.0613  0.0508   0.0726 0.0   0.0

2. DID LOCALISATION CHANGE?
   class_name   n  auroc_base  auroc_ft  delta_obs  ci_low  ci_high      p   p_bh
 Cardiomegaly 107       0.459    0.4183    -0.0407 -0.0464  -0.0349 0.0000 0.0000
Consolidation  43       0.671    0.6863     0.0153  0.0038   0.0265 0.0184 0.0184

3. CALIBRATION TEST -- observed vs pre-registered prediction
   class_name   n  slope   pred     obs  residual  res_ci_low  res_ci_high  corr_pred_obs  p_resi

In [ ]:
import json, os, shutil, subprocess
from pathlib import Path

# ---------------- config ----------------
WORKING       = Path("/kaggle/working")
DATASET_SLUG  = "ext_lora_rank8_ft_evaluation"      # lowercase, hyphens, 3-50 chars
DATASET_TITLE = "Ext LoRA rank 8 ft evaluation"   # 6-50 chars
STAGING       = Path("/kaggle/_ds_staging")    # OUTSIDE /kaggle/working on purpose
# ----------------------------------------

# 1. credentials from Secrets
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
username = sec.get_secret("KAGGLE_USERNAME")
key      = sec.get_secret("KAGGLE_KEY")

kdir = Path.home() / ".kaggle"; kdir.mkdir(exist_ok=True)
(kdir / "kaggle.json").write_text(json.dumps({"username": username, "key": key}))
(kdir / "kaggle.json").chmod(0o600)
os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = username, key
print(f"credentials set for {username}")

# 2. copy EVERYTHING from /kaggle/working
if STAGING.exists():
    shutil.rmtree(STAGING)
shutil.copytree(WORKING, STAGING)

# drop junk that shouldn't ship
for pattern in ["**/__pycache__", "**/.ipynb_checkpoints", "**/.git"]:
    for p in STAGING.glob(pattern):
        shutil.rmtree(p, ignore_errors=True)

total = 0
print(f"\nstaged from {WORKING}:")
for p in sorted(STAGING.rglob("*")):
    if p.is_file():
        mb = p.stat().st_size / 1e6
        total += mb
        print(f"  {p.relative_to(STAGING)}  {mb:.2f} MB")
print(f"\ntotal: {total:.1f} MB  ({sum(1 for p in STAGING.rglob('*') if p.is_file())} files)")
assert total > 0, "/kaggle/working is empty"

# 3. metadata
meta = {"title": DATASET_TITLE,
        "id": f"{username}/{DATASET_SLUG}",
        "licenses": [{"name": "CC0-1.0"}]}
(STAGING / "dataset-metadata.json").write_text(json.dumps(meta, indent=2))
print(f"id: {meta['id']}")

# 4. create, or version if it already exists
def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout or "", r.stderr or "")
    return r.returncode

if run(f'kaggle datasets create -p "{STAGING}" --dir-mode zip') != 0:
    print("create failed — trying as a new version")
    run(f'kaggle datasets version -p "{STAGING}" -m "step3 lora + manifest" --dir-mode zip')

print(f"\nhttps://www.kaggle.com/datasets/{username}/{DATASET_SLUG}")